In [5]:
# ═══ CELL 1 — Configuration & API Setup ═══

# ═══════════════════════════════════════════════════════════
# VERSEPULSE CONFIGURATION
# ═══════════════════════════════════════════════════════════
#
# DEMO MODE is ON by default — no API keys needed.
# When you receive your hackathon API credentials,
# follow the two steps marked with 🔑 below.
#
# ═══════════════════════════════════════════════════════════

YOUVERSION_API_KEY = ""   # 🔑 Paste your YouVersion API key here
GLOO_AI_API_KEY    = ""   # 🔑 Paste your Gloo AI Studio key here

YOUVERSION_API_BASE = "https://api.youversion.com/v1"
GLOO_AI_API_BASE    = "https://api.gloo.ai/studio/v1"

DEMO_MODE = True   # 🔑 Change to False when you have real API keys

if DEMO_MODE:
    print("=" * 55)
    print("  VersePulse running in DEMO MODE")
    print("  All Scripture served locally. No API keys needed.")
    print("=" * 55)
else:
    if not YOUVERSION_API_KEY or not GLOO_AI_API_KEY:
        raise ValueError(
            "DEMO_MODE is False but one or both API keys are missing.\n"
            "Either add your keys above or set DEMO_MODE = True."
        )
    print("=" * 55)
    print("  VersePulse running in LIVE MODE")
    print("  Connected to YouVersion + Gloo AI Studio APIs.")
    print("=" * 55)


# ═══ CELL 2 — Install & Import Dependencies ═══

import json
import random
import textwrap
from dataclasses import dataclass
from typing import Optional

try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    REQUESTS_AVAILABLE = False
    print("Note: 'requests' not found. Demo mode will still work.")
    print("To install: pip install requests")

print("Libraries loaded successfully.")


# ═══ CELL 3 — Data Models ═══

@dataclass
class BiometricSnapshot:
    """
    Represents a single moment of biometric data
    captured from a wearable device.
    """
    heart_rate: int
    hr_zone: int
    effort_pct: float
    recovery_score: int
    stress_index: float
    activity_type: str
    session_minute: int
    user_translation: str = "NIV"
    user_language_tag: str = "en"

@dataclass
class VerseDelivery:
    """
    The final package delivered to the wearable display.
    """
    reference: str
    text: str
    translation: str
    moment_type: str
    delivery_format: str
    encouragement_note: str

print("Data models ready.")


# ═══ CELL 4 — Local Scripture Library ═══

VERSE_LIBRARY = {
    "PHI.4.13": {
        "NIV": "I can do all this through him who gives me strength.",
        "ESV": "I can do all things through him who strengthens me.",
        "KJV": "I can do all things through Christ which strengtheneth me.",
        "NLT": "For I can do everything through Christ, who gives me strength."
    },
    "ISA.40.31": {
        "NIV": "But those who hope in the LORD will renew their strength. "
               "They will soar on wings like eagles; they will run and not "
               "grow weary, they will walk and not be faint.",
        "ESV": "But they who wait for the LORD shall renew their strength; "
               "they shall mount up with wings like eagles.",
        "KJV": "But they that wait upon the LORD shall renew their strength.",
        "NLT": "But those who trust in the LORD will find new strength."
    },
    "ROM.8.37": {
        "NIV": "No, in all these things we are more than conquerors "
               "through him who loved us.",
        "ESV": "No, in all these things we are more than conquerors "
               "through him who loved us.",
        "KJV": "Nay, in all these things we are more than conquerors "
               "through him that loved us.",
        "NLT": "No, despite all these things, overwhelming victory is "
               "ours through Christ, who loved us."
    },
    "PSA.46.10": {
        "NIV": "He says, 'Be still, and know that I am God.'",
        "ESV": "Be still, and know that I am God.",
        "KJV": "Be still, and know that I am God.",
        "NLT": "Be still, and know that I am God."
    },
    "GAL.6.9": {
        "NIV": "Let us not become weary in doing good, for at the proper "
               "time we will reap a harvest if we do not give up.",
        "ESV": "And let us not grow weary of doing good, for in due season "
               "we will reap, if we do not give up.",
        "KJV": "And let us not be weary in well doing: for in due season "
               "we shall reap, if we faint not.",
        "NLT": "So let's not get tired of doing what is good. At just the "
               "right time we will reap a harvest of blessing."
    },
    "ISA.41.10": {
        "NIV": "So do not fear, for I am with you; do not be dismayed, "
               "for I am your God. I will strengthen you and help you.",
        "ESV": "Fear not, for I am with you; be not dismayed, for I am "
               "your God; I will strengthen you.",
        "KJV": "Fear thou not; for I am with thee: be not dismayed; "
               "for I am thy God: I will strengthen thee.",
        "NLT": "Don't be afraid, for I am with you. Don't be discouraged, "
               "for I am your God. I will strengthen you and help you."
    },
    "JOS.1.9": {
        "NIV": "Have I not commanded you? Be strong and courageous. "
               "Do not be afraid; do not be discouraged, for the LORD "
               "your God will be with you wherever you go.",
        "ESV": "Have I not commanded you? Be strong and courageous. "
               "Do not be frightened, and do not be dismayed.",
        "KJV": "Have not I commanded thee? Be strong and of a good courage.",
        "NLT": "This is my command — be strong and courageous! "
               "Do not be afraid or discouraged."
    },
    "2TI.1.7": {
        "NIV": "For the Spirit God gave us does not make us timid, but gives "
               "us power, love and self-discipline.",
        "ESV": "For God gave us a spirit not of fear but of power and love "
               "and self-control.",
        "KJV": "For God hath not given us the spirit of fear; but of power, "
               "and of love, and of a sound mind.",
        "NLT": "For God has not given us a spirit of fear and timidity, "
               "but of power, love, and self-discipline."
    },
    "1CO.9.24": {
        "NIV": "Do you not know that in a race all the runners run, but only "
               "one gets the prize? Run in such a way as to get the prize.",
        "ESV": "Do you not know that in a race all the runners run, but only "
               "one receives the prize? So run that you may obtain it.",
        "KJV": "Know ye not that they which run in a race run all, but one "
               "receiveth the prize? So run, that ye may obtain.",
        "NLT": "Don't you realize that in a race everyone runs, but only one "
               "person gets the prize? So run to win!"
    },
    "PSA.118.24": {
        "NIV": "The LORD has done it this very day; let us rejoice today "
               "and be glad.",
        "ESV": "This is the day that the LORD has made; let us rejoice "
               "and be glad in it.",
        "KJV": "This is the day which the LORD hath made; we will rejoice "
               "and be glad in it.",
        "NLT": "This is the day the LORD has made. We will rejoice and "
               "be glad in it."
    },
    "LAM.3.22": {
        "NIV": "Because of the LORD's great love we are not consumed, for "
               "his compassions never fail. They are new every morning.",
        "ESV": "The steadfast love of the LORD never ceases; his mercies "
               "never come to an end; they are new every morning.",
        "KJV": "It is of the LORD's mercies that we are not consumed, "
               "because his compassions fail not.",
        "NLT": "The faithful love of the LORD never ends! His mercies "
               "never cease. Great is his faithfulness."
    },
    "PSA.23.4": {
        "NIV": "Even though I walk through the darkest valley, I will fear "
               "no evil, for you are with me; your rod and your staff, "
               "they comfort me.",
        "ESV": "Even though I walk through the valley of the shadow of death,"
               " I will fear no evil, for you are with me.",
        "KJV": "Yea, though I walk through the valley of the shadow of death,"
               " I will fear no evil: for thou art with me.",
        "NLT": "Even when I walk through the darkest valley, I will not be "
               "afraid, for you are close beside me."
    },
    "PRO.3.5": {
        "NIV": "Trust in the LORD with all your heart and lean not on your "
               "own understanding.",
        "ESV": "Trust in the LORD with all your heart, and do not lean on "
               "your own understanding.",
        "KJV": "Trust in the LORD with all thine heart; and lean not unto "
               "thine own understanding.",
        "NLT": "Trust in the LORD with all your heart; do not depend on "
               "your own understanding."
    },
    "2CO.12.9": {
        "NIV": "But he said to me, 'My grace is sufficient for you, for my "
               "power is made perfect in weakness.'",
        "ESV": "But he said to me, 'My grace is sufficient for you, for my "
               "power is made perfect in weakness.'",
        "KJV": "And he said unto me, My grace is sufficient for thee: for "
               "my strength is made perfect in weakness.",
        "NLT": "Each time he said, 'My grace is all you need. My power works "
               "best in weakness.'"
    },
}

def get_verse_text(reference: str, translation: str) -> str:
    verse = VERSE_LIBRARY.get(reference, {})
    return (
        verse.get(translation)
        or verse.get("NIV")
        or "The LORD is my strength and my shield."
    )

print(f"Scripture library loaded: {len(VERSE_LIBRARY)} verses")
print(f"Translations available: NIV, ESV, KJV, NLT")


# ═══ CELL 5 — Moment & Verse Mapping ═══

MOMENT_TO_VERSE_MAP = {
    "peak_effort":       [("ISA.40.31", "endurance"), ("ROM.8.37",  "victory")],
    "breakthrough_wall": [("PHI.4.13",  "strength"),  ("JOS.1.9",   "courage")],
    "final_rep":         [("2CO.12.9",  "grace"),     ("PHI.4.13",  "strength")],
    "steady_state":      [("PSA.23.4",  "presence"),  ("PRO.3.5",   "trust")],
    "recovery_window":   [("PSA.46.10", "peace"),     ("LAM.3.22",  "renewal")],
    "rest_set":          [("LAM.3.22",  "renewal"),   ("PSA.46.10", "peace")],
    "finishing_strong":  [("GAL.6.9",   "perseverance"),("1CO.9.24","purpose")],
    "redline":           [("ISA.41.10", "courage"),   ("ROM.8.37",  "victory")],
    "pre_workout":       [("PSA.118.24","gratitude"),  ("JOS.1.9",  "courage")],
    "post_workout":      [("1CO.9.24",  "purpose"),   ("GAL.6.9",  "perseverance")],
    "early_push":        [("JOS.1.9",   "courage"),   ("2TI.1.7",   "power")],
    "warmup":            [("PRO.3.5",   "trust"),     ("PSA.118.24","gratitude")],
    "active_recovery":   [("LAM.3.22",  "renewal"),   ("PSA.23.4",  "presence")],
}

DELIVERY_FORMAT_MAP = {
    "peak_effort":       "haptic_pulse + display",
    "breakthrough_wall": "haptic_pulse + audio",
    "final_rep":         "haptic_pulse + display",
    "redline":           "haptic_pulse + audio",
    "finishing_strong":  "haptic_pulse + display",
    "recovery_window":   "ambient_glow",
    "rest_set":          "display_only",
    "steady_state":      "display_only",
    "pre_workout":       "display_only",
    "post_workout":      "display_only",
    "early_push":        "display_only",
    "warmup":            "ambient_glow",
    "active_recovery":   "ambient_glow",
}

DEMO_ENCOURAGEMENT_MAP = {
    "breakthrough_wall": "This is the moment you trained for.",
    "peak_effort":       "Everything you have — leave it here.",
    "final_rep":         "One more. That's all it takes.",
    "recovery_window":   "Rest is part of the work.",
    "rest_set":          "Breathe. Reload. Go again.",
    "finishing_strong":  "You decided to finish before you started.",
    "redline":           "Pain is temporary. This moment is not.",
    "pre_workout":       "Today was given to you on purpose.",
    "post_workout":      "Well done — body and soul.",
    "early_push":        "Set the tone now. Own the rest.",
    "steady_state":      "Find your rhythm and hold it.",
    "warmup":            "Prepare well. Perform well.",
    "active_recovery":   "Let your body catch up to your heart.",
}

print("Moment maps loaded.")
print(f"Moment types supported: {len(MOMENT_TO_VERSE_MAP)}")


# ═══ CELL 6 — Core Classification Engine ═══

def classify_moment(bio: BiometricSnapshot) -> str:
    if bio.session_minute == 0:
        return "pre_workout"
    if bio.hr_zone == 5 and bio.effort_pct >= 0.95:
        return "redline"
    if bio.hr_zone == 5 and bio.effort_pct >= 0.85:
        if bio.activity_type == "weightlifting":
            return "final_rep"
        return "peak_effort"
    if bio.hr_zone == 4 and bio.effort_pct >= 0.75:
        if bio.session_minute <= 5:
            return "early_push"
        if bio.activity_type == "running" and bio.session_minute >= 4:
            return "breakthrough_wall"
        return "breakthrough_wall"
    if bio.hr_zone in [2, 3] and bio.effort_pct < 0.65:
        if bio.activity_type == "weightlifting":
            return "rest_set"
        if bio.activity_type == "hiit":
            return "active_recovery"
        return "steady_state"
    if bio.hr_zone <= 2 and bio.stress_index < 2.5:
        return "recovery_window"
    if bio.effort_pct >= 0.80 and bio.session_minute >= 18:
        return "finishing_strong"
    if bio.session_minute == 0 and bio.hr_zone == 1:
        return "warmup"
    return "steady_state"


def select_verse(moment_type: str, session_minute: int) -> str:
    options = MOMENT_TO_VERSE_MAP.get(
        moment_type, [("PHI.4.13", "strength")]
    )
    index = session_minute % len(options)
    return options[index][0]

print("Classifier ready.")


# ═══ CELL 7 — API Layer (Live + Demo) ═══

def fetch_verse(reference: str, translation: str,
                language_tag: str) -> dict:
    if DEMO_MODE:
        return {
            "reference": reference,
            "text": get_verse_text(reference, translation),
            "translation": translation,
            "source": "local_library"
        }
    else:
        headers = {
            "Authorization": f"Bearer {YOUVERSION_API_KEY}",
            "Accept": "application/json"
        }
        params = {
            "reference": reference,
            "version_abbreviation": translation,
            "language_tag": language_tag
        }
        try:
            response = requests.get(
                f"{YOUVERSION_API_BASE}/verse",
                headers=headers,
                params=params,
                timeout=5
            )
            if response.status_code == 200:
                return response.json()
            else:
                print(f"YouVersion API returned {response.status_code}."
                      " Falling back to local library.")
                return {
                    "reference": reference,
                    "text": get_verse_text(reference, translation),
                    "translation": translation,
                    "source": "fallback"
                }
        except Exception as e:
            print(f"YouVersion API error: {e}. Using local fallback.")
            return {
                "reference": reference,
                "text": get_verse_text(reference, translation),
                "translation": translation,
                "source": "fallback"
            }


def generate_encouragement(verse_text: str, verse_reference: str,
                            moment_type: str, activity_type: str,
                            bio: BiometricSnapshot) -> str:
    if DEMO_MODE:
        return DEMO_ENCOURAGEMENT_MAP.get(
            moment_type, "You were made for moments like this."
        )
    else:
        headers = {
            "Authorization": f"Bearer {GLOO_AI_API_KEY}",
            "Content-Type": "application/json"
        }
        prompt = (
            f"A person is {activity_type} at "
            f"{bio.effort_pct*100:.0f}% effort in HR zone {bio.hr_zone}. "
            f"Moment type: {moment_type}. "
            f'Scripture: "{verse_text}" — {verse_reference}. '
            f"Write one brief, grounded encouragement sentence "
            f"connecting this verse to their physical moment. "
            f"Under 15 words. No emojis. Not preachy."
        )
        payload = {
            "model": "gloo-faith-v1",
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "You are a faith-aware fitness coach. "
                        "Brief, natural, grounded. Never preachy."
                    )
                },
                {"role": "user", "content": prompt}
            ],
            "max_tokens": 40,
            "temperature": 0.7,
            "safety_filter": "ministry_safe"
        }
        try:
            response = requests.post(
                f"{GLOO_AI_API_BASE}/chat/completions",
                headers=headers,
                json=payload,
                timeout=5
            )
            if response.status_code == 200:
                return (response.json()["choices"][0]
                        ["message"]["content"].strip())
            else:
                print(f"Gloo AI returned {response.status_code}."
                      " Using local encouragement.")
                return DEMO_ENCOURAGEMENT_MAP.get(
                    moment_type, "You were made for moments like this."
                )
        except Exception as e:
            print(f"Gloo AI error: {e}. Using local encouragement.")
            return DEMO_ENCOURAGEMENT_MAP.get(
                moment_type, "You were made for moments like this."
            )

print("API layer ready.")
print(f"Mode: {'DEMO (local data)' if DEMO_MODE else 'LIVE (real APIs)'}")


# ═══ CELL 8 — Main Pipeline ═══

def process_biometric_snapshot(bio: BiometricSnapshot) -> VerseDelivery:
    print("\n" + "─" * 45)
    print(f"  Activity : {bio.activity_type.title()}")
    print(f"  HR       : {bio.heart_rate} bpm  |  Zone {bio.hr_zone}")
    print(f"  Effort   : {bio.effort_pct*100:.0f}%")
    print(f"  Minute   : {bio.session_minute}")
    print("─" * 45)

    moment_type  = classify_moment(bio)
    print(f"  ✦ Moment     : {moment_type.replace('_', ' ').title()}")

    verse_ref    = select_verse(moment_type, bio.session_minute)
    print(f"  ✦ Reference  : {verse_ref}")

    verse_data   = fetch_verse(
        verse_ref, bio.user_translation, bio.user_language_tag
    )
    verse_text   = verse_data.get("text", "")
    print(f"  ✦ Source     : {verse_data.get('source', 'api')}")

    encouragement = generate_encouragement(
        verse_text, verse_ref, moment_type,
        bio.activity_type, bio
    )

    return VerseDelivery(
        reference=verse_ref,
        text=verse_text,
        translation=bio.user_translation,
        moment_type=moment_type,
        delivery_format=DELIVERY_FORMAT_MAP.get(
            moment_type, "display_only"
        ),
        encouragement_note=encouragement
    )


def display_delivery(delivery: VerseDelivery):
    width = 55
    print("\n" + "═" * width)
    print("  VERSEPULSE DELIVERY".center(width))
    print("═" * width)
    print(f"  Moment     : "
          f"{delivery.moment_type.replace('_', ' ').title()}")
    print(f"  Reference  : {delivery.reference} ({delivery.translation})")
    print()
    wrapped = textwrap.fill(delivery.text, width=width - 4)
    for line in wrapped.split("\n"):
        print(f"  {line}")
    print()
    print(f"  ❝ {delivery.encouragement_note} ❞")
    print()
    print(f"  Delivery   : {delivery.delivery_format}")
    print("═" * width)

print("Pipeline ready.")


# ═══ CELL 9 — Run Demo Scenarios ═══

scenarios = [
    BiometricSnapshot(
        heart_rate=162, hr_zone=4, effort_pct=0.78,
        recovery_score=72, stress_index=3.2,
        activity_type="running", session_minute=4,
        user_translation="NIV"
    ),
    BiometricSnapshot(
        heart_rate=178, hr_zone=5, effort_pct=0.94,
        recovery_score=61, stress_index=5.0,
        activity_type="cycling", session_minute=6,
        user_translation="ESV"
    ),
    BiometricSnapshot(
        heart_rate=158, hr_zone=4, effort_pct=0.81,
        recovery_score=84, stress_index=4.2,
        activity_type="weightlifting", session_minute=3,
        user_translation="KJV"
    ),
    BiometricSnapshot(
        heart_rate=132, hr_zone=2, effort_pct=0.51,
        recovery_score=61, stress_index=2.1,
        activity_type="cycling", session_minute=9,
        user_translation="NIV"
    ),
    BiometricSnapshot(
        heart_rate=188, hr_zone=5, effort_pct=0.97,
        recovery_score=55, stress_index=5.8,
        activity_type="hiit", session_minute=15,
        user_translation="NLT"
    ),
    BiometricSnapshot(
        heart_rate=165, hr_zone=4, effort_pct=0.82,
        recovery_score=91, stress_index=4.4,
        activity_type="running", session_minute=22,
        user_translation="NIV"
    ),
]

print("Running all demo scenarios...\n")

results = []
for i, snapshot in enumerate(scenarios, 1):
    print(f"SCENARIO {i} of {len(scenarios)}")
    delivery = process_biometric_snapshot(snapshot)
    display_delivery(delivery)
    results.append(delivery)

print(f"\n✓ {len(results)} scenarios completed successfully.")


# ═══ CELL 10 — Results Summary Table ═══

print("\n" + "═" * 75)
print("  SESSION SUMMARY".center(75))
print("═" * 75)
print(
    f"  {'#':<4}"
    f"{'Activity':<16}"
    f"{'Moment':<22}"
    f"{'Verse':<12}"
    f"{'Translation':<6}"
)
print("─" * 75)

for i, (scenario, delivery) in enumerate(zip(scenarios, results), 1):
    moment_display = delivery.moment_type.replace("_", " ").title()
    print(
        f"  {i:<4}"
        f"{scenario.activity_type.title():<16}"
        f"{moment_display:<22}"
        f"{delivery.reference:<12}"
        f"{delivery.translation:<6}"
    )

print("═" * 75)
print(f"\n  Mode: {'DEMO — local data' if DEMO_MODE else 'LIVE — real APIs'}")
print(f"  To switch to live APIs:")
print(f"  1. Add your API keys in Cell 1 where marked with 🔑")
print(f"  2. Set DEMO_MODE = False in Cell 1")
print(f"  3. Re-run all cells")
print("═" * 75)


  VersePulse running in DEMO MODE
  All Scripture served locally. No API keys needed.
Libraries loaded successfully.
Data models ready.
Scripture library loaded: 14 verses
Translations available: NIV, ESV, KJV, NLT
Moment maps loaded.
Moment types supported: 13
Classifier ready.
API layer ready.
Mode: DEMO (local data)
Pipeline ready.
Running all demo scenarios...

SCENARIO 1 of 6

─────────────────────────────────────────────
  Activity : Running
  HR       : 162 bpm  |  Zone 4
  Effort   : 78%
  Minute   : 4
─────────────────────────────────────────────
  ✦ Moment     : Early Push
  ✦ Reference  : JOS.1.9
  ✦ Source     : local_library

═══════════════════════════════════════════════════════
                   VERSEPULSE DELIVERY                 
═══════════════════════════════════════════════════════
  Moment     : Early Push
  Reference  : JOS.1.9 (NIV)

  Have I not commanded you? Be strong and courageous.
  Do not be afraid; do not be discouraged, for the
  LORD your God will be 